# PH2 External Validation: Melanoma vs. Non-Melanoma

This notebook prepares PH2 labels, extracts features using the Swin-B, EVA-02, and CoAtNet backbones, and evaluates a binary transfer-learning ensemble trained on ISIC 2019 data.

**Task definition:** `0 = Non-Melanoma` (common and atypical nevi), `1 = Melanoma`.

## Execution order
1. Verify that the PH2 dataset is already available in the Colab runtime.
2. Export dermoscopic images to the working image directory.
3. Build the binary label CSV from the PH2 metadata workbook.
4. Extract Swin-B, EVA-02, and CoAtNet features.
5. Run the ISIC-to-PH2 external transfer evaluation.

## Dataset and reproducibility notes
- This notebook does not download the dataset. Place the PH2 dataset in Colab before running the first setup cell.
- By default, the expected layout is `/content/PH2Dataset/PH2 Dataset images/` and `/content/PH2Dataset/PH2_dataset.xlsx`. If your files are elsewhere, set `PH2_ROOT`, `PH2_IMAGES_SOURCE_DIR`, or `PH2_METADATA_XLSX` in the runtime environment before continuing.
- Do not commit the PH2 dataset, checkpoint weights, feature arrays, prediction arrays, or local output files to GitHub.
- The PH2 external labels must only be used for final evaluation; threshold, calibration, and fusion parameters are selected using the internal ISIC folds.
- This notebook has been statically inspected, but the complete pipeline has not been run end-to-end in a clean GPU environment.
- Review feature-row alignment, binary label mapping, lesion-group construction, and checkpoint-loading reports before interpreting results.


## 1. Verify the PH2 Dataset Already Exists in Colab

This setup cell assumes that the PH2 dataset has already been uploaded to or mounted in the Colab runtime. It does not download files or request authentication tokens.

Expected default layout:

```text
/content/PH2Dataset/
├── PH2_dataset.xlsx
└── PH2 Dataset images/
    └── .../*_Dermoscopic_Image/*.bmp
```

If your dataset uses a different location, configure `PH2_ROOT`, `PH2_IMAGES_SOURCE_DIR`, and/or `PH2_METADATA_XLSX` before running this cell.


In [ ]:
import os
import glob
from pathlib import Path

# Configure these paths if the PH2 dataset is stored elsewhere in Colab.
PH2_ROOT = Path(os.environ.get("PH2_ROOT", "/content/PH2Dataset"))
PH2_IMAGES_SOURCE_DIR = Path(
    os.environ.get("PH2_IMAGES_SOURCE_DIR", str(PH2_ROOT / "PH2 Dataset images"))
)
PH2_METADATA_XLSX = Path(
    os.environ.get("PH2_METADATA_XLSX", str(PH2_ROOT / "PH2_dataset.xlsx"))
)

# Stop early with a clear message if the expected files are not present.
if not PH2_IMAGES_SOURCE_DIR.is_dir():
    raise FileNotFoundError(
        f"PH2 image directory was not found: {PH2_IMAGES_SOURCE_DIR}. "
        "Upload or mount the PH2 dataset in Colab, then update PH2_IMAGES_SOURCE_DIR."
    )

if not PH2_METADATA_XLSX.is_file():
    raise FileNotFoundError(
        f"PH2 metadata workbook was not found: {PH2_METADATA_XLSX}. "
        "Upload or mount the workbook in Colab, then update PH2_METADATA_XLSX."
    )

# Make the paths available to the following notebook cells.
os.environ["PH2_ROOT"] = str(PH2_ROOT)
os.environ["PH2_IMAGES_SOURCE_DIR"] = str(PH2_IMAGES_SOURCE_DIR)
os.environ["PH2_METADATA_XLSX"] = str(PH2_METADATA_XLSX)

image_paths = glob.glob(
    str(PH2_IMAGES_SOURCE_DIR / "**" / "*_Dermoscopic_Image" / "*.bmp"),
    recursive=True,
)
if not image_paths:
    raise FileNotFoundError(
        "No dermoscopic BMP files were found under "
        f"{PH2_IMAGES_SOURCE_DIR}. Check the folder structure or update the path."
    )

print("PH2 dataset found in the Colab runtime.")
print(f"PH2 image directory: {PH2_IMAGES_SOURCE_DIR}")
print(f"PH2 metadata workbook: {PH2_METADATA_XLSX}")
print(f"Dermoscopic BMP images found: {len(image_paths)}")


## 2. Export Dermoscopic Images

The next cell scans the PH2 image directory that is already present in Colab and converts the dermoscopic BMP images to PNG files in a single working directory. It uses `PH2_IMAGES_SOURCE_DIR` configured in the setup cell.


In [ ]:
import os
import cv2
import glob

base_dir = os.environ.get("PH2_IMAGES_SOURCE_DIR", "/content/PH2Dataset/PH2 Dataset images")
save_dir = "/content/ph2_images"

os.makedirs(save_dir, exist_ok=True)

# Scan only dermoscopic image directories.
paths = glob.glob(base_dir + "/**/*_Dermoscopic_Image/*.bmp", recursive=True)

count = 0

for path in paths:
    img = cv2.imread(path)

    name = os.path.basename(path).replace(".bmp", ".png")

    cv2.imwrite(os.path.join(save_dir, name), img)

    count += 1

print("Total dermoscopic images:", count)

## 3. Create Binary PH2 Labels

Read the PH2 metadata workbook and map `Melanoma` to label 1 and `Common Nevus`/`Atypical Nevus` to label 0. The resulting CSV is shared by all feature-extraction steps.

Before accepting this label file, inspect the diagnosis counts and verify that every image has one recognized diagnosis. The current source code is preserved; see the review notes for the validation checks recommended before reporting results.

In [ ]:
import pandas as pd

print("----- READING PH2 METADATA -----")

metadata_xlsx = os.environ.get("PH2_METADATA_XLSX", "/content/PH2Dataset/PH2_dataset.xlsx")
df = pd.read_excel(metadata_xlsx, header=12)

df = df[df["Image Name"].notna()]

print("Total records:", len(df))

print("----- DIAGNOSIS COUNTS BEFORE LABEL MAPPING -----")

df["diagnosis"] = None

df.loc[df["Melanoma"] == "X", "diagnosis"] = "Melanoma"
df.loc[df["Atypical Nevus"] == "X", "diagnosis"] = "Atypical Nevus"
df.loc[df["Common Nevus"] == "X", "diagnosis"] = "Common Nevus"

print(df["diagnosis"].value_counts())

print("----- BINARY LABEL COUNTS -----")

df["binary_label"] = df["diagnosis"].replace({
    "Common Nevus": "non-melanoma",
    "Atypical Nevus": "non-melanoma",
    "Melanoma": "melanoma"
})

print(df["binary_label"].value_counts())

print("----- SAVING LABEL CSV -----")

df["image_name"] = df["Image Name"] + ".png"

df_out = df[["image_name", "diagnosis", "binary_label"]]

labels_csv = os.environ.get("PH2_LABELS_CSV", "/content/ph2_labels_binary.csv")
df_out.to_csv(labels_csv, index=False)

print(f"Saved labels to: {labels_csv}")

## 4. Swin-B Feature Extraction

Extract 1,024-dimensional Swin-B features from PH2 images at 768 × 768 resolution and save the corresponding binary labels. Features and labels are emitted in dataset iteration order.

In [ ]:
# ==============================================================================
# PH2 — FEATURE EXTRACTION (SWIN 768, 1024-dim)
# ==============================================================================

import os
import torch
import timm
import numpy as np
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
from tqdm.auto import tqdm

# ----------------------------------------------------------------------
# CONFIG
# ----------------------------------------------------------------------
MODEL_CKPT = os.environ.get("SWIN_PH2_CHECKPOINT", "/content/checkpoints/swin/best_model_swin768.pth")

IMAGE_DIR = os.environ.get("PH2_IMAGE_DIR", "/content/ph2_images")
META_CSV  = os.environ.get("PH2_LABELS_CSV", "/content/ph2_labels_binary.csv")

OUT_DIR = "/content/PH2_features"
os.makedirs(OUT_DIR, exist_ok=True)

FEAT_NPY = os.path.join(OUT_DIR, "swin_ph2_features.npy")
META_NPY = os.path.join(OUT_DIR, "swin_ph2_meta.npy")

IMAGE_SIZE = (768, 768)
BATCH_SIZE = 64
NUM_WORKERS = 4
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("DEVICE:", DEVICE)

# ----------------------------------------------------------------------
# TRANSFORM
# ----------------------------------------------------------------------
transform = transforms.Compose([
    transforms.Resize(IMAGE_SIZE),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])

# ----------------------------------------------------------------------
# DATASET
# ----------------------------------------------------------------------
class PH2Dataset(Dataset):
    def __init__(self, csv_path, image_dir, transform=None):
        self.df = pd.read_csv(csv_path)
        self.image_dir = image_dir
        self.transform = transform

        # Convert the binary label to numeric values.
        self.df["melanoma"] = (self.df["binary_label"] == "melanoma").astype(int)

        print("Total samples:", len(self.df))
        print("Melanoma class distribution:\n", self.df["melanoma"].value_counts())

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = os.path.join(self.image_dir, row["image_name"])
        label = int(row["melanoma"])

        img = Image.open(img_path).convert("RGB")
        if self.transform:
            img = self.transform(img)

        return img, label

# ----------------------------------------------------------------------
# MODEL (Swin 768, 1024-dim)
# ----------------------------------------------------------------------
print("\nLoading model...")
model = timm.create_model(
    "swin_base_patch4_window12_384_in22k",
    pretrained=False,
    num_classes=0,
    img_size=768
)

state = torch.load(MODEL_CKPT, map_location="cpu")
model.load_state_dict(state, strict=False)

model.to(DEVICE)
model.eval()
print("Model is ready.")

# ----------------------------------------------------------------------
# LOADER
# ----------------------------------------------------------------------
dataset = PH2Dataset(META_CSV, IMAGE_DIR, transform)
loader = DataLoader(dataset, batch_size=BATCH_SIZE,
                    shuffle=False, num_workers=NUM_WORKERS)

# ----------------------------------------------------------------------
# FEATURE EXTRACTION
# ----------------------------------------------------------------------
N = len(dataset)
feat_dim = 1024

X = np.empty((N, feat_dim), dtype=np.float32)
y = np.empty(N, dtype=np.int64)

ptr = 0

print("\n[PH2] Starting feature extraction...")
with torch.no_grad():
    for imgs, labels in tqdm(loader):
        bsz = imgs.size(0)
        imgs = imgs.to(DEVICE)

        feats = model.forward_head(
            model.forward_features(imgs),
            pre_logits=True
        )

        feats = feats.float().cpu().numpy()

        X[ptr:ptr+bsz] = feats
        y[ptr:ptr+bsz] = labels.numpy()

        ptr += bsz

np.save(FEAT_NPY, X)
np.save(META_NPY, y)

print("\nFeature extraction completed.")
print("Feature shape:", X.shape)

## 5. EVA-02 Feature Extraction

Extract 768-dimensional EVA-02 features from PH2 images at 672 × 672 resolution. Review the checkpoint's missing/unexpected key counts: `strict=False` can allow loading to complete even when some weights were not loaded.

In [ ]:
# ==============================================================================
# PH2 — EVA-02 FEATURE EXTRACTION (A100 FAST)
# - Input: 672x672
# - Output: (N, 768) float32
# - Label: malignant (0/1)
# ==============================================================================

import os
import torch
import timm
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from torchvision.io import read_image
import torchvision.transforms.functional as TF

# ------------------------------------------------------------------------------
# CONFIG
# ------------------------------------------------------------------------------
EVA_ROOT = os.environ.get("EVA_CHECKPOINT_DIR", "/content/models/eva02")
CKPT_STAGE2 = os.path.join(EVA_ROOT, "checkpoints", "best_model_stage2_finetuned.pth")
CKPT_STAGE1 = os.path.join(EVA_ROOT, "checkpoints", "best_model_stage1_main.pth")

MODEL_CKPT = CKPT_STAGE2 if os.path.exists(CKPT_STAGE2) else CKPT_STAGE1
assert os.path.exists(MODEL_CKPT), "EVA checkpoint was not found."

IMG_DIR = os.environ.get("PH2_IMAGE_DIR", "/content/ph2_images")
CSV_PATH = os.environ.get("PH2_LABELS_CSV", "/content/ph2_labels_binary.csv")

OUT_DIR = os.path.join("/content/PH2_features/")
os.makedirs(OUT_DIR, exist_ok=True)

FEAT_NPY = os.path.join(OUT_DIR, "eva_ph2_features.npy")
META_NPY = os.path.join(OUT_DIR, "eva_ph2_meta.npy")

IMG_SIZE = 672
BATCH_SIZE = 128
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

torch.backends.cudnn.benchmark = True
torch.set_float32_matmul_precision("high")

print("DEVICE:", DEVICE)
print("Checkpoint:", MODEL_CKPT)

# ------------------------------------------------------------------------------
# LOAD CSV
# ------------------------------------------------------------------------------
df = pd.read_csv(CSV_PATH)

assert "image_name" in df.columns, "❌ image_name kolonu yok!"
assert "binary_label" in df.columns, "❌ binary_label kolonu yok!"

df["melanoma"] = (df["binary_label"] == "melanoma").astype(np.int8)

ids = df["image_name"].astype(str).values
labels = df["melanoma"].values
N = len(ids)

print("Total samples:", N)
print("Melanoma class distribution:\n", df["melanoma"].value_counts())

# ------------------------------------------------------------------------------
# MODEL
# ------------------------------------------------------------------------------
model = timm.create_model(
    "eva02_base_patch14_448.mim_in22k_ft_in1k",
    pretrained=False,
    num_classes=0,
    img_size=IMG_SIZE
)

state = torch.load(MODEL_CKPT, map_location="cpu")
missing, unexpected = model.load_state_dict(state, strict=False)
print(f"Checkpoint loaded | missing={len(missing)} | unexpected={len(unexpected)}")

model.to(DEVICE).eval()
model = model.to(memory_format=torch.channels_last)

# ------------------------------------------------------------------------------
# WARM-UP
# ------------------------------------------------------------------------------
dummy = torch.zeros(1, 3, IMG_SIZE, IMG_SIZE, device=DEVICE)
with torch.no_grad(), torch.amp.autocast("cuda", dtype=torch.bfloat16):
    _ = model.forward_head(model.forward_features(dummy), pre_logits=True)
del dummy
torch.cuda.synchronize()

# ------------------------------------------------------------------------------
# PRE-ALLOC
# ------------------------------------------------------------------------------
X = np.empty((N, 768), dtype=np.float32)
y = np.empty(N, dtype=np.int8)

mean = torch.tensor([0.485, 0.456, 0.406], device=DEVICE).view(1,3,1,1)
std  = torch.tensor([0.229, 0.224, 0.225], device=DEVICE).view(1,3,1,1)

# ------------------------------------------------------------------------------
# MAIN LOOP
# ------------------------------------------------------------------------------
ptr = 0
print("\n[PH2] Starting EVA-02 feature extraction...")

for i in tqdm(range(0, N, BATCH_SIZE)):
    batch_ids = ids[i:i+BATCH_SIZE]
    batch_lbl = labels[i:i+BATCH_SIZE]
    bsz = len(batch_ids)

    imgs = []

    for img_name in batch_ids:
        img_path = os.path.join(IMG_DIR, img_name)

        img = read_image(img_path)

        if img.shape[0] == 4:
            img = img[:3]

        img = img.float() / 255.0
        img = TF.resize(img, [IMG_SIZE, IMG_SIZE])

        imgs.append(img)

    imgs = torch.stack(imgs).to(DEVICE, non_blocking=True)
    imgs = (imgs - mean) / std
    imgs = imgs.to(memory_format=torch.channels_last)

    with torch.no_grad(), torch.amp.autocast("cuda", dtype=torch.bfloat16):
        feats = model.forward_head(
            model.forward_features(imgs),
            pre_logits=True
        )

    feats = feats.float().cpu().numpy()

    X[ptr:ptr+bsz] = feats
    y[ptr:ptr+bsz] = batch_lbl
    ptr += bsz

assert ptr == N, f"Counter mismatch: {ptr} != {N}"

# ------------------------------------------------------------------------------
# SAVE
# ------------------------------------------------------------------------------
np.save(FEAT_NPY, X)
np.save(META_NPY, y)

print("\nEVA-02 PH2 feature extraction completed.")
print("Features:", X.shape)
print("Labels :", y.shape)
print("Output directory:", OUT_DIR)

## 6. CoAtNet Feature Extraction

Extract CoAtNet features at 672 × 672 resolution. The checkpoint loader can resize selected relative-position bias tables and skip incompatible parameters. Review the `resized`, `skipped`, and `missing` counts before treating the features as belonging to the intended fine-tuned checkpoint.

In [ ]:
import os, math, torch, timm, numpy as np, pandas as pd
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from tqdm.auto import tqdm
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

# ================= CONFIG =================

CKPT = os.environ.get("COATNET_PH2_CHECKPOINT", "/content/checkpoints/coatnet/best_model_ft_672.pth")

CSV  = os.environ.get("PH2_LABELS_CSV", "/content/ph2_labels_binary.csv")
IMG_DIR = "/content/ph2_images"

OUT_DIR = "/content/PH2_features"
os.makedirs(OUT_DIR, exist_ok=True)

DEVICE="cuda" if torch.cuda.is_available() else "cpu"
IMG=672
BATCH=128
WORKERS=4

torch.backends.cudnn.benchmark=True

print("Device:", DEVICE)

# =========================================================
# MODEL
# =========================================================

class CoAtNetWithAux(nn.Module):
    def __init__(self,name):
        super().__init__()
        self.backbone=timm.create_model(
            name,
            pretrained=False,
            num_classes=0,
            global_pool="avg",
            img_size=IMG
        )
    def forward(self,x):
        return self.backbone(x)

MODEL_NAME="coatnet_1_rw_224"

model=CoAtNetWithAux(MODEL_NAME).to(DEVICE).eval()

# =========================================================
# RELPOS RESIZE
# =========================================================

def sqrt_int(n):
    s=int(round(math.sqrt(n)))
    return s if s*s==n else None

def resize_relpos(src,target_tokens):
    L_old,heads=src.shape
    old=sqrt_int(L_old)
    new=sqrt_int(target_tokens)
    if old is None or new is None:
        return None
    w=src.T.reshape(heads,old,old)[None]
    w=F.interpolate(w,size=(new,new),mode="bicubic",align_corners=False)
    w=w.squeeze(0).reshape(heads,new*new).T
    return w

# =========================================================
# LOAD CHECKPOINT SAFE
# =========================================================

print("Loading checkpoint...")

ckpt=torch.load(CKPT,map_location="cpu")

clean={}
for k,v in ckpt.items():
    if k.startswith("module."):
        k=k[7:]
    clean[k]=v

target=model.state_dict()

resized=0
skipped=0

for k in list(clean.keys()):

    if k not in target:
        del clean[k]; skipped+=1; continue

    if clean[k].shape==target[k].shape:
        continue

    if "relative_position_bias_table" in k:
        new=resize_relpos(clean[k],target[k].shape[0])
        if new is not None:
            clean[k]=new
            resized+=1
        else:
            del clean[k]; skipped+=1
    else:
        del clean[k]; skipped+=1

load=model.load_state_dict(clean,strict=False)

print("Checkpoint loaded")
print("resized:",resized)
print("skipped:",skipped)
print("missing:",len(load.missing_keys))

# =========================================================
# DATASET (PH2)
# =========================================================

tfm=transforms.Compose([
    transforms.Resize((IMG,IMG)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],
                         [0.229,0.224,0.225])
])

class PH2(Dataset):
    def __init__(self):
        df=pd.read_csv(CSV)

        df["melanoma"]=(df["binary_label"]=="melanoma").astype(int)

        self.ids=df["image_name"].astype(str).values
        self.labels=df["melanoma"].astype(int).values

        print("Samples:",len(self.ids))
        print("Melanoma class distribution:\n", df["melanoma"].value_counts())

    def __len__(self):
        return len(self.ids)

    def __getitem__(self,i):
        img_name=self.ids[i]
        img_path=os.path.join(IMG_DIR,img_name)

        img=Image.open(img_path).convert("RGB")

        return tfm(img), img_name, self.labels[i]

ds=PH2()

loader=DataLoader(
    ds,
    batch_size=BATCH,
    shuffle=False,
    num_workers=WORKERS,
    pin_memory=True
)

# =========================================================
# FEATURE EXTRACTION
# =========================================================

N=len(ds)
feat_dim=model.backbone.num_features

print("Feature dim:",feat_dim)

X=np.empty((N,feat_dim),dtype=np.float32)
meta=np.empty((N,2),dtype=object)

ptr=0

print("\nExtracting features...")

with torch.no_grad():
    for imgs,ids,lbls in tqdm(loader):

        imgs=imgs.to(DEVICE,non_blocking=True)

        with torch.amp.autocast("cuda",
                                enabled=(DEVICE=="cuda")):
            feats=model(imgs)

        feats=feats.float().cpu().numpy()

        b=feats.shape[0]

        X[ptr:ptr+b]=feats
        meta[ptr:ptr+b,0]=ids
        meta[ptr:ptr+b,1]=lbls.numpy()

        ptr+=b

assert ptr==N

# =========================================================
# SAVE
# =========================================================

np.save(os.path.join(OUT_DIR,"coat_ph2_features.npy"),X)
np.save(os.path.join(OUT_DIR,"coat_ph2_meta.npy"),meta[:,1])

print("\nDONE")
print("shape:",X.shape)
print("saved to:",OUT_DIR)

## 7. ISIC-to-PH2 External Transfer Evaluation

Train binary classifiers and learn calibration/fusion parameters inside five-fold stratified group cross-validation on the combined ISIC train and validation splits. The resulting fold-specific models generate PH2 probabilities, which are averaged for external evaluation.

The melanoma threshold is optimized on pooled internal OOF predictions, not on PH2 labels. PH2 labels are used to calculate final external metrics and class-stratified bootstrap confidence intervals.

**Interpretation:** This is a binary transfer experiment, where PH2 common and atypical nevi are grouped as Non-Melanoma. Confirm feature-row alignment, class mapping, lesion-group construction, and checkpoint loading before interpreting the results.

In [ ]:
# ============================================================
#  B12_FINAL_PH2_EXTERNAL_MELANOMA_TRANSFER.py
#  - Pipeline     : StratifiedGroupKFold (ISIC Train+Val) -> OOF Fold-Ensembling
#  - Task         : Binary Transfer Evaluation (MELANOMA vs NON-MELANOMA)
#  - Calibration  : Two-Stage (T_i step=0.02, T_ens step=0.005)
#  - Fusion       : Strict NLL-optimal Weight Search
#  - Decision     : Standard Argmax (0.5) vs OOF BAcc-Optimized Threshold
#  - Reproducibility: STRICT DETERMINISM
#  - Deliverables : 95% CI Bootstrapped Metrics & Class-wise Reports (PH2)
# ============================================================

import os
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

import sys, platform, json, warnings, copy
from pathlib import Path
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, f1_score, log_loss,
    roc_auc_score, roc_curve, auc, confusion_matrix, precision_recall_fscore_support
)
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression

import torch, torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

# --- STRICT REPRODUCIBILITY ---
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False

# -------------------- Configuration and Paths --------------------
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
RANDOM_STATE = 42
N_SPLITS = 5

CLASS_NAMES = ["Non-Melanoma", "Melanoma"]

# --- ISIC (Internal) Paths ---
SPLIT_DIR = os.environ.get("ISIC_SPLIT_DIR", "/content/isic2019/splits")
ISIC_FEAT_DIRS = {
    "SWIN": os.environ.get("ISIC_SWIN_FEATURE_DIR", "/content/isic2019/features/swin"),
    "EVA" : os.environ.get("ISIC_EVA_FEATURE_DIR", "/content/isic2019/features/eva02"),
    "COAT": os.environ.get("ISIC_COAT_FEATURE_DIR", "/content/isic2019/features/coatnet"),
}
META_CSV = os.environ.get("ISIC_METADATA_CSV", "/content/isic2019/metadata/ISIC_2019_Training_Metadata.csv")

# --- PH2 (External) Paths ---
PH2_FEAT_DIRS = {
    "SWIN": os.environ.get("PH2_SWIN_FEATURES", "/content/PH2_features/swin_ph2_features.npy"),
    "EVA" : os.environ.get("PH2_EVA_FEATURES", "/content/PH2_features/eva_ph2_features.npy"),
    "COAT": os.environ.get("PH2_COAT_FEATURES", "/content/PH2_features/coat_ph2_features.npy"),
}
# PH2 labels: 0 = Non-Melanoma (common/atypical nevus), 1 = Melanoma; expected shape: (N,).
PH2_META_Y = os.environ.get("PH2_LABELS_NPY", "/content/PH2_features/swin_ph2_meta.npy")

BASE_SAVE_DIR = os.environ.get("PH2_TRANSFER_OUTPUT_DIR", "/content/outputs/ph2_melanoma_transfer")
os.makedirs(BASE_SAVE_DIR, exist_ok=True)

USE_PCA = True
PCA_VARIANCE = 0.99
T_MIN, T_MAX, T_STEP = 0.70, 1.50, 0.02
STAGE2_T_STEP = 0.005
MLP_BATCH = 256
COAT_FLOOR = 0.05
W_STEP = 0.02
N_CALIB_BINS = 15
N_BOOTSTRAP = 1000
FIG_DPI = 600

NAMED_PRESETS = {
    "SwinMLP": {"hidden":[2048], "dropout":0.16057, "lr":3.414e-4, "weight_decay":1.488e-4, "epochs":70},
    "EvaMLP":  {"hidden":[2048], "dropout":0.32033, "lr":9.765e-4, "weight_decay":8.212e-5, "epochs":70},
}
HP_LOGREG = {"C":2.34, "max_iter":587, "tol":3.6e-04, "solver":"lbfgs", "class_weight":"balanced", "n_jobs":-1, "random_state":RANDOM_STATE}
COAT_C_GRID = [1.6, HP_LOGREG["C"], 3.2]

def set_seed():
    import random
    random.seed(RANDOM_STATE); np.random.seed(RANDOM_STATE)
    torch.manual_seed(RANDOM_STATE); torch.cuda.manual_seed_all(RANDOM_STATE)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    torch.use_deterministic_algorithms(True, warn_only=False)

# -------------------- Data Loading and Validation --------------------
# Binary mapping: Melanoma vs. Non-Melanoma.
def map_to_melanoma(class_name: str) -> int:
    return 1 if str(class_name).upper() == "MEL" else 0

def _glob_any(p, names):
    for nm in names:
        if (Path(p) / nm).exists(): return str(Path(p) / nm)
    return None

def _normalize_ids(arr):
    return pd.Series(arr, dtype=str).str.replace(r"^.*[\\/]", "", regex=True).str.replace(r"\.(jpg|jpeg|png)$", "", regex=True).values

def load_melanoma_transfer_data():
    df_tr = pd.read_csv(os.path.join(SPLIT_DIR, "train_split.csv"))
    df_va = pd.read_csv(os.path.join(SPLIT_DIR, "val_split.csv"))
    df_int = pd.concat([df_tr, df_va], ignore_index=True)

    id_col = next(c for c in ["image_id","image","filename"] if c in df_int.columns)
    ids_int = _normalize_ids(df_int[id_col])

    if "class_name" in df_int.columns:
        y_int = df_int["class_name"].apply(map_to_melanoma).values
    else:
        raise ValueError("The class_name column is missing; MEL vs. non-Melanoma mapping cannot be performed.")

    # Load PH2 labels.
    y_ext = np.load(PH2_META_Y)

    data = {}
    for feat in ["SWIN", "EVA", "COAT"]:
        X1 = np.load(_glob_any(ISIC_FEAT_DIRS[feat], ["train_features.npy", "features_train.npy", "train.npy"]))
        X2 = np.load(_glob_any(ISIC_FEAT_DIRS[feat], ["val_features.npy", "features_val.npy", "val.npy"]))
        Xext = np.load(PH2_FEAT_DIRS[feat])

        # Sanity checks for feature counts.
        if len(X1) + len(X2) != len(ids_int):
            raise ValueError(f"ERROR: {feat} ISIC feature count ({len(X1) + len(X2)}) does not match the number of metadata rows.")
        if len(Xext) != len(y_ext):
            raise ValueError(f"ERROR: {feat} PH2 feature count ({len(Xext)}) does not match the number of PH2 labels.")

        data[feat] = {"Xint": np.vstack([X1, X2]), "Xext": Xext}

    meta_df = pd.read_csv(META_CSV)
    meta_df["_img"] = _normalize_ids(meta_df[id_col].astype(str).values) if id_col in meta_df else meta_df.index
    lesion_col = next((c for c in ["lesion_id", "lesion"] if c in meta_df.columns), None)
    meta_df["_lesion_id"] = meta_df[lesion_col].astype(str) if lesion_col else meta_df["_img"]
    meta_map = meta_df.set_index("_img")["_lesion_id"].to_dict()
    lesion_ids = np.array([meta_map.get(i, f"unknown_{i}") for i in ids_int])

    print("\n[SANITY CHECK PASSED] Feature counts and binary labels (MEL vs. non-Melanoma) were checked.")
    print(f"ISIC (Transfer OOF): {len(y_int)} (Melanoma: {np.sum(y_int==1)})")
    print(f"PH2 (External Test): {len(y_ext)} (Melanoma: {np.sum(y_ext==1)})")

    return ids_int, y_int, lesion_ids, y_ext, data

# -------------------- Modeller --------------------
class RobustMLP(nn.Module):
    def __init__(self, in_dim, n_classes=2, hp=None):
        super().__init__()
        layers = [nn.Linear(in_dim, hp["hidden"][0]), nn.BatchNorm1d(hp["hidden"][0]), nn.ReLU(), nn.Dropout(hp["dropout"]), nn.Linear(hp["hidden"][0], n_classes)]
        self.net = nn.Sequential(*layers)
    def forward(self, x): return self.net(x)

def train_mlp_and_predict(Xtr, ytr, Xval, Xte, hp, device):
    mdl = RobustMLP(Xtr.shape[1], 2, hp).to(device)
    opt = torch.optim.AdamW(mdl.parameters(), lr=hp["lr"], weight_decay=hp["weight_decay"])
    crit = nn.CrossEntropyLoss(label_smoothing=0.05).to(device)

    g = torch.Generator(device="cpu"); g.manual_seed(RANDOM_STATE)
    loader = DataLoader(TensorDataset(torch.tensor(Xtr, dtype=torch.float32), torch.tensor(ytr, dtype=torch.long)), batch_size=MLP_BATCH, shuffle=True, generator=g)

    mdl.train()
    for _ in range(hp["epochs"]):
        for xb, yb in loader:
            opt.zero_grad(set_to_none=True); crit(mdl(xb.to(device)), yb.to(device)).backward(); opt.step()

    def infer(X):
        mdl.eval()
        with torch.no_grad(): return torch.softmax(mdl(torch.tensor(X, dtype=torch.float32).to(device)), 1).cpu().numpy()
    return infer(Xval), infer(Xte)

def pca_pipe(Xtr, Xva, Xte):
    sc = StandardScaler().fit(Xtr)
    Ztr, Zva, Zte = sc.transform(Xtr), sc.transform(Xva), sc.transform(Xte)
    pca_t = PCA(n_components=min(Ztr.shape)-1, svd_solver="full", random_state=RANDOM_STATE).fit(Ztr)
    k = np.argmax(np.cumsum(pca_t.explained_variance_ratio_) >= PCA_VARIANCE) + 1
    pca = PCA(n_components=k, svd_solver="full", random_state=RANDOM_STATE).fit(Ztr)
    return pca.transform(Ztr), pca.transform(Zva), pca.transform(Zte)

# -------------------- Calibration and Threshold Optimization --------------------
def temp_scale(P, T):
    log_Q = np.log(np.clip(P, 1e-15, 1.0)) / T
    Q = np.exp(log_Q - np.max(log_Q, axis=1, keepdims=True))
    return Q / Q.sum(axis=1, keepdims=True)

def pick_T_by_nll(y_true, P, n_class, step):
    grid = np.arange(T_MIN, T_MAX+1e-9, step)
    losses = [log_loss(y_true, temp_scale(P, t), labels=range(n_class)) for t in grid]
    return grid[np.argmin(losses)]

def weight_search_nll(P_dict, y_true, n_class=2):
    best = {"score": 1e18, "ws":0.33, "we":0.33, "wc":0.34}
    for ws in np.arange(0.0, 1.01, W_STEP):
        for we in np.arange(0.0, 1.01, W_STEP):
            wc = 1.0 - ws - we
            if wc < COAT_FLOOR - 1e-9: continue
            score = log_loss(y_true, np.clip(ws*P_dict["SWIN"] + we*P_dict["EVA"] + wc*P_dict["COAT"], 1e-15, 1.0), labels=range(n_class))
            if score < best["score"]: best.update({"score":score, "ws":ws, "we":we, "wc":wc})
    return best

# Fine-grained grid for binary threshold optimization.
def optimize_binary_threshold_bacc(y_true, P):
    best_t, best_score = 0.5, 0.0
    for t in np.linspace(0.001, 0.999, 250):
        y_pred = (P[:, 1] >= t).astype(int)
        score = balanced_accuracy_score(y_true, y_pred)
        if score > best_score:
            best_score, best_t = score, t
    return best_t

# -------------------- Reporting Utilities --------------------
def compute_ece(y_true, P):
    conf = np.max(P, 1); acc = (np.argmax(P, 1) == y_true)
    bins = np.linspace(0, 1, N_CALIB_BINS+1)
    return sum(np.abs(np.mean(conf[m]) - np.mean(acc[m])) * np.mean(m) for i in range(N_CALIB_BINS) if np.sum(m := ((conf > bins[i]) & (conf <= bins[i+1])) if i>0 else ((conf >= 0) & (conf <= bins[1]))) > 0)

def apply_rule(P, thr=None):
    return P.argmax(1) if thr is None else (P[:, 1] >= thr).astype(int)

def eval_prob_metrics(y_true, P):
    P_c = np.clip(P, 1e-15, 1.0)
    Y_ohe = np.eye(2)[y_true]
    return {
        "ROC_AUC": float(roc_auc_score(y_true, P_c[:, 1])),
        "ECE": compute_ece(y_true, P_c),
        "NLL": float(log_loss(y_true, P_c, labels=[0, 1])),
        "Brier": float(np.mean(np.sum((P_c - Y_ohe) ** 2, axis=1)))
    }

def eval_class_metrics(y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    sens = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    spec = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    f1_macro = f1_score(y_true, y_pred, average='macro', zero_division=0)
    f1_weighted = f1_score(y_true, y_pred, average='weighted', zero_division=0)

    return {
        "Accuracy": float(accuracy_score(y_true, y_pred)),
        "Balanced_Accuracy": float(balanced_accuracy_score(y_true, y_pred)),
        "Macro_F1": float(f1_macro),
        "Weighted_F1": float(f1_weighted),
        "Sensitivity_Melanoma": float(sens),
        "Specificity_NonMelanoma": float(spec),
        "Precision_Melanoma": float(prec)
    }

# Per-class specificity for the binary confusion matrix.
def build_per_class_report(y_true, P, y_pred):
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    prec, rec, f1, sup = precision_recall_fscore_support(y_true, y_pred, labels=[0, 1], zero_division=0)
    tn, fp, fn, tp = cm.ravel()

    specs = [
        tp / (tp + fn) if (tp + fn) > 0 else 0.0, # Non-Melanoma Specificity = Melanoma Sensitivity
        tn / (tn + fp) if (tn + fp) > 0 else 0.0  # Melanoma Specificity = Non-Melanoma Sensitivity (TN)
    ]

    rows = []
    for k, cname in enumerate(CLASS_NAMES):
        rows.append({"Class": cname, "ROC_AUC": float(roc_auc_score(y_true==k, P[:, k])),
                     "Precision": float(prec[k]), "Sensitivity": float(rec[k]),
                     "Specificity": float(specs[k]), "F1_Score": float(f1[k]), "Support": int(sup[k])})

    w = sup / np.maximum(sup.sum(), 1)
    rows.append({"Class": "Macro_Avg", "ROC_AUC": float(roc_auc_score(y_true, P[:, 1])),
                 "Precision": np.mean(prec), "Sensitivity": np.mean(rec),
                 "Specificity": np.mean(specs), "F1_Score": np.mean(f1), "Support": int(sup.sum())})
    rows.append({"Class": "Weighted_Avg", "ROC_AUC": float(roc_auc_score(y_true, P[:, 1])),
                 "Precision": np.sum(prec*w), "Sensitivity": np.sum(rec*w),
                 "Specificity": np.sum(specs*w), "F1_Score": np.sum(f1*w), "Support": int(sup.sum())})
    return pd.DataFrame(rows)

def bootstrap_full_evaluation(y_true, P, thr_bacc, n_boot=N_BOOTSTRAP):
    rng = np.random.default_rng(RANDOM_STATE)
    Pc = np.clip(P, 1e-15, 1.0)
    idx_list = [np.where(y_true == k)[0] for k in range(2)]

    pt_prob = eval_prob_metrics(y_true, Pc)
    pt_std = eval_class_metrics(y_true, apply_rule(Pc, None))
    pt_opt_bacc = eval_class_metrics(y_true, apply_rule(Pc, thr_bacc))
    pt_delta_bacc = pt_opt_bacc["Balanced_Accuracy"] - pt_std["Balanced_Accuracy"]

    boots = {"prob": {k: np.zeros(n_boot) for k in pt_prob}, "std": {k: np.zeros(n_boot) for k in pt_std},
             "opt_bacc": {k: np.zeros(n_boot) for k in pt_opt_bacc}, "delta_bacc": np.zeros(n_boot)}

    for b in range(n_boot):
        idx = np.concatenate([rng.choice(c_idx, size=len(c_idx), replace=True) for c_idx in idx_list if len(c_idx)>0])
        yb, pb = y_true[idx], Pc[idx]

        m_prob = eval_prob_metrics(yb, pb)
        m_std = eval_class_metrics(yb, apply_rule(pb, None))
        m_bacc = eval_class_metrics(yb, apply_rule(pb, thr_bacc))

        for k in pt_prob: boots["prob"][k][b] = m_prob[k]
        for k in pt_std: boots["std"][k][b], boots["opt_bacc"][k][b] = m_std[k], m_bacc[k]
        boots["delta_bacc"][b] = m_bacc["Balanced_Accuracy"] - m_std["Balanced_Accuracy"]

    def fmt(pt, bt, rule):
        rows = []
        for k, p in pt.items():
            arr = bt[k][~np.isnan(bt[k])]
            lo, hi = (np.percentile(arr, 2.5), np.percentile(arr, 97.5)) if len(arr)>0 else (np.nan, np.nan)
            rows.append({"Decision Rule": rule, "Metric": k, "Estimate": p, "CI95_Low": lo, "CI95_High": hi, "Formatted_95CI": f"{p:.4f} [{lo:.4f}, {hi:.4f}]"})
        return rows

    res_prob = fmt(pt_prob, boots["prob"], "Decision-Free (Probabilities)")
    res_class = fmt(pt_std, boots["std"], "1. Standard Argmax") + fmt(pt_opt_bacc, boots["opt_bacc"], "2. OOF BAcc-Optimized Rule")

    arr_delta = boots["delta_bacc"][~np.isnan(boots["delta_bacc"])]
    lo, hi = (np.percentile(arr_delta, 2.5), np.percentile(arr_delta, 97.5)) if len(arr_delta)>0 else (np.nan, np.nan)
    res_class.append({"Decision Rule": "Comparison", "Metric": "Delta_BAcc", "Estimate": pt_delta_bacc, "CI95_Low": lo, "CI95_High": hi, "Formatted_95CI": f"{pt_delta_bacc:+.4f} [{lo:+.4f}, {hi:+.4f}]"})

    return pd.DataFrame(res_class), pd.DataFrame(res_prob)

# -------------------- Main Experiment Loop --------------------
def main():
    set_seed()
    ids_int, y_int, lesion_ids, y_ext, data = load_melanoma_transfer_data()

    print(f"\n{'='*80}\n🚀 STARTING BINARY TRANSFER EVALUATION (ISIC -> PH2)\nTask: Melanoma vs. Non-Melanoma\n{'='*80}")

    OOF_P = np.zeros((len(y_int), 2))
    EXT_P_Folds = np.zeros((N_SPLITS, len(y_ext), 2))

    sgkf = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

    for fold, (tr_idx, va_idx) in enumerate(sgkf.split(ids_int, y_int, groups=lesion_ids)):
        print(f"\n   🔄 [FOLD {fold+1}/{N_SPLITS}] Training & Calibration...")
        y_tr, y_va = y_int[tr_idx], y_int[va_idx]

        P_val, P_ext, T_sel = {}, {}, {}
        for m in ["SWIN", "EVA", "COAT"]:
            Xtr, Xva = data[m]["Xint"][tr_idx], data[m]["Xint"][va_idx]
            Xte = data[m]["Xext"]

            Ztr, Zva, Zte = pca_pipe(Xtr, Xva, Xte)

            if m in ["SWIN", "EVA"]:
                P_va_raw, P_te_raw = train_mlp_and_predict(Ztr, y_tr, Zva, Zte, NAMED_PRESETS[f"{m.capitalize()}MLP"], DEVICE)
                T_i = pick_T_by_nll(y_va, P_va_raw, 2, step=T_STEP)
                P_val[m], P_ext[m], T_sel[m] = temp_scale(P_va_raw, T_i), temp_scale(P_te_raw, T_i), T_i
            else:
                best_nll, best_Pva, best_Pte, best_T = 1e18, None, None, None
                for C in COAT_C_GRID:
                    lg = LogisticRegression(C=float(C), **{k:v for k,v in HP_LOGREG.items() if k!="C"}).fit(Ztr, y_tr)
                    P_va_raw, P_te_raw = lg.predict_proba(Zva), lg.predict_proba(Zte)
                    Tc = pick_T_by_nll(y_va, P_va_raw, 2, step=T_STEP)
                    if (nll := log_loss(y_va, np.clip(temp_scale(P_va_raw, Tc), 1e-15, 1.0))) < best_nll:
                        best_nll, best_Pva, best_Pte, best_T = nll, temp_scale(P_va_raw, Tc), temp_scale(P_te_raw, Tc), Tc
                P_val["COAT"], P_ext["COAT"], T_sel["COAT"] = best_Pva, best_Pte, best_T

        w = weight_search_nll(P_val, y_va, 2)
        P_ens_va = w["ws"]*P_val["SWIN"] + w["we"]*P_val["EVA"] + w["wc"]*P_val["COAT"]
        T_ens = pick_T_by_nll(y_va, P_ens_va, 2, step=STAGE2_T_STEP)

        OOF_P[va_idx] = temp_scale(P_ens_va, T_ens)
        EXT_P_Folds[fold] = temp_scale(w["ws"]*P_ext["SWIN"] + w["we"]*P_ext["EVA"] + w["wc"]*P_ext["COAT"], T_ens)

    print("\n[STEP 3] Optimizing the binary decision threshold on pooled internal OOF predictions...")
    opt_thr_bacc = optimize_binary_threshold_bacc(y_int, OOF_P)
    print(f"   -> OOF BAcc Threshold (\u03C4) for Melanoma: {opt_thr_bacc:.3f}")

    print("\n[STEP 4] Ensembling PH2 external predictions and generating reports...")
    P_final_ext = np.mean(EXT_P_Folds, axis=0)

    tab1_class, tab4_prob = bootstrap_full_evaluation(y_ext, P_final_ext, opt_thr_bacc, N_BOOTSTRAP)
    tab_cls_std = build_per_class_report(y_ext, P_final_ext, apply_rule(P_final_ext, None))
    tab_cls_opt = build_per_class_report(y_ext, P_final_ext, apply_rule(P_final_ext, opt_thr_bacc))

    tab1_class.to_csv(os.path.join(BASE_SAVE_DIR, f"Table1_PH2_Transfer_ClassificationRules.csv"), index=False)
    tab4_prob.to_csv(os.path.join(BASE_SAVE_DIR, f"Table4_PH2_Transfer_ProbCalibration.csv"), index=False)
    tab_cls_std.to_csv(os.path.join(BASE_SAVE_DIR, f"Table2_PH2_Classwise_Argmax.csv"), index=False)
    tab_cls_opt.to_csv(os.path.join(BASE_SAVE_DIR, f"Table2_PH2_Classwise_OptBAcc.csv"), index=False)
    np.save(os.path.join(BASE_SAVE_DIR, f"y_proba_ph2_external.npy"), P_final_ext)
    np.save(os.path.join(BASE_SAVE_DIR, f"y_proba_OOF_isic_internal.npy"), OOF_P)

    print("\n--- [TABLE 1: PH2 EXTERNAL TRANSFER DECISION RULES] ---")
    print(tab1_class.groupby(["Decision Rule", "Metric"])[["Formatted_95CI"]].first().to_string())
    print(f"\nPH2 evaluation completed. Outputs: {BASE_SAVE_DIR}")

if __name__ == "__main__":
    main()